In [ ]:
import subprocess
try:
    print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print('no GPU (nvidia-smi not found)')

In [ ]:
%pip uninstall -y -q torchao
%pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors matplotlib "peft>=0.11.0" "trl>=0.9.0" "bitsandbytes>=0.43.0"

In [ ]:
import pathlib, os, sys, shutil

for root, dirs, files in os.walk('/kaggle/input'):
    depth = root.count(os.sep) - '/kaggle/input'.count(os.sep)
    if depth <= 3:
        print(root)
    if depth >= 4:
        dirs[:] = []

matches = [p.parent.parent for p in pathlib.Path('/kaggle/input').rglob('src/config.py')
           if (p.parent.parent / 'scripts' / 'train_sft.py').exists() and '/project/' not in str(p)]
assert matches, 'repo dataset (src/config.py + scripts/) not found under /kaggle/input'
repo_src = matches[0]
print('repo dataset found at', repo_src)

proj = pathlib.Path('/kaggle/working/project')
if not proj.exists():
    shutil.copytree(repo_src, proj)
os.chdir(proj)
sys.path.insert(0, str(proj))
os.environ['PYTHONPATH'] = str(proj)
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import subprocess
def run(cmd):
    print('$', cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f'command failed with exit code {proc.returncode}: {cmd}')

import importlib
for name in ('torch', 'transformers', 'datasets', 'peft', 'trl'):
    try:
        print(name, importlib.import_module(name).__version__)
    except Exception as error:
        print(name, 'unavailable:', error)

## Locate checkpoints and the alpaca-eval subset file from kernel_sources

In [ ]:
import glob, shutil, os
sft_cands = glob.glob('/kaggle/input/**/project/outputs/sft/sft_run/checkpoints/best', recursive=True)
dpo_hh_cands = glob.glob('/kaggle/input/**/project/outputs/dpo_hh_rlhf/hh_rlhf_run/checkpoints/best', recursive=True)
dpo_uf_cands = glob.glob('/kaggle/input/**/project/outputs/dpo_ultrafeedback/ultrafeedback_run/checkpoints/best', recursive=True)
alpaca_cands = glob.glob('/kaggle/input/**/project/eval_sets/alpaca_eval_300_seed42.json', recursive=True)
assert sft_cands and dpo_hh_cands and dpo_uf_cands and alpaca_cands, (sft_cands, dpo_hh_cands, dpo_uf_cands, alpaca_cands)
sft_ckpt = sft_cands[0]
dpo_hh_ckpt = dpo_hh_cands[0]
dpo_uf_ckpt = dpo_uf_cands[0]
os.makedirs('eval_sets', exist_ok=True)
shutil.copy(alpaca_cands[0], 'eval_sets/alpaca_eval_300_seed42.json')
print('sft_ckpt', sft_ckpt)
print('dpo_hh_ckpt', dpo_hh_ckpt)
print('dpo_uf_ckpt', dpo_uf_ckpt)

## Task 6 -- generation + pairwise file, HH-RLHF track

In [ ]:
run(f"python scripts/generate_eval_outputs.py --eval_set eval_sets/hh_rlhf_eval_300_seed42.json --sft_checkpoint {sft_ckpt} --dpo_checkpoint {dpo_hh_ckpt} --dpo_track hh_rlhf --output_dir outputs/eval")

## Task 6 -- generation + pairwise file, UltraFeedback track (AlpacaEval prompts)

In [ ]:
run(f"python scripts/generate_eval_outputs.py --eval_set eval_sets/alpaca_eval_300_seed42.json --sft_checkpoint {sft_ckpt} --dpo_checkpoint {dpo_uf_ckpt} --dpo_track ultrafeedback --output_dir outputs/eval")

## Confirm outputs

In [ ]:
import pathlib
for p in sorted(pathlib.Path('outputs/eval').rglob('*')):
    if p.is_file():
        print(p)